In [1]:
!git clone --single-branch --branch VanHung https://github.com/hoangminhtit/vn-hackaithon.git
# !git clone --single-branch --branch main https://github.com/hoangminhtit/vn-hackaithon.git -- nhánh main

fatal: destination path 'vn-hackaithon' already exists and is not an empty directory.


In [2]:
# Cell 2: Cài dependencies (llama-cpp-python + huggingface_hub)
!pip install -r  vn-hackaithon/requirements.txt

In [5]:
# Cell 2b: Cài llama-cpp-python bản CUDA (compile từ source, ~10-15 phút)
# Bắt buộc để llama.cpp thật sự dùng GPU — chạy 1 lần duy nhất
import torch
if torch.cuda.is_available():
    print(f"✅ CUDA {torch.version.cuda} / {torch.cuda.get_device_name(0)} — compiling llama-cpp-python với CUDA...")
    !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --force-reinstall --no-cache-dir -q
    print("✅ llama-cpp-python CUDA ready — restart kernel trước khi chạy pipeline!")
else:
    print("ℹ️ Không có GPU — bỏ qua, dùng bản CPU")

✅ CUDA 12.8 / Tesla T4 — compiling llama-cpp-python với CUDA...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 294.3 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 190.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 343.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 235.0 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 195.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
ERROR: ERROR: Failed to buil

In [4]:
# Cell 3: Cấu hình env vars
import os

os.environ["HF_TOKEN"] = "<YOUR_HF_TOKEN>"
os.environ["HF_MODEL_ID"] = "unsloth/Qwen3.5-4B-GGUF"
os.environ["GGUF_FILE"] = "Qwen3.5-4B-Q4_K_M.gguf"
os.environ["HF_LOCAL_DIR"] = "model"
os.environ["LLM_MAX_NEW_TOKENS"] = "16"
os.environ["LLM_ANSWER_MAX_TOKENS"] = "16"
os.environ["LLM_USE_LLM_ROUTE"] = "0"
os.environ["RAG_MAX_CONTEXT_CHARS"] = "12000"
os.environ["RAG_FULL_PASSAGE_CHARS"] = "12000"
os.environ["RAG_BM25_MAX_CHARS"] = "10000"
# LLAMA_N_GPU_LAYERS: code tự detect CUDA/Metal/CPU — không cần set thủ công.
# Muốn ép CPU:  os.environ["LLAMA_N_GPU_LAYERS"] = "0"
os.environ["LLAMA_N_GPU_LAYERS"] = "-1"

In [12]:
# Cell 4: Tải GGUF model (~2.7 GB cho Q4_K_M)
!python vn-hackaithon/download_model.py

✅ Đã có sẵn: /kaggle/working/model/Qwen3.5-4B-Q4_K_M.gguf (2614 MB)


In [13]:
# Cell 5: Kiểm tra model đã tải
!ls -lh model/*.gguf

-rw-r--r-- 1 root root 2.6G Jun 11 09:07 model/Qwen3.5-4B-Q4_K_M.gguf


In [ ]:
# Cell 6: Chạy pipeline
!python vn-hackaithon/predict.py \
  --input "/kaggle/input/datasets/vanhung23668291/vn-hackaithon/public-test_1780368312.json" \
  --output-dir "output/" \
  --mode llm

llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized
